# 1.2 Code Brief: Build Regularized Logistic Regression Models

Quick reference for building L1, L2, and ElasticNet regularized models.

## Setup

In [ ]:
import pandas as pd
import pickle
import os
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder

## Load Data

In [ ]:
# Set up file paths
data_filepath = '../data/'
models_filepath = '../models/'

df_training = pd.read_csv(f'{data_filepath}training.csv')
X_train = df_training
y_train = df_training['SEM_3_STATUS']

## Define Feature Groups

In [ ]:
minmax_columns = ['HS_GPA', 'GPA_1', 'GPA_2', 'DFW_RATE_1', 'DFW_RATE_2']
standard_columns = ['UNITS_ATTEMPTED_1', 'UNITS_ATTEMPTED_2']
categorical_columns = ['GENDER', 'RACE_ETHNICITY', 'FIRST_GEN_STATUS']

## Create Preprocessor

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('minmax', MinMaxScaler(), minmax_columns),
        ('standard', StandardScaler(), standard_columns),
        ('onehot', OneHotEncoder(handle_unknown='ignore', 
                                  drop=['Female', 'Other', 'Unknown'], 
                                  sparse_output=False), categorical_columns)
    ],
    remainder='drop'
)

## Build L2 (Ridge) Model

In [ ]:
model_l2 = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(
        penalty='l2',
        C=1.0,
        class_weight='balanced',
        solver='liblinear',
        random_state=42
    ))
])

## Build L1 (Lasso) Model

In [ ]:
model_l1 = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(
        penalty='l1',
        C=1.0,
        class_weight='balanced',
        solver='saga',  # Required for L1
        random_state=42
    ))
])

## Build ElasticNet Model

In [ ]:
model_elasticnet = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(
        penalty='elasticnet',
        C=1.0,
        l1_ratio=0.5,  # 50% L1, 50% L2
        class_weight='balanced',
        solver='saga',  # Required for ElasticNet
        random_state=42
    ))
])

## Save Models

In [ ]:
models_dict = {
    "L2 Ridge": model_l2,
    "L1 Lasso": model_l1,
    "ElasticNet": model_elasticnet
}

models_path = f'{models_filepath}'
os.makedirs(models_path, exist_ok=True)

for name, model_pipeline in models_dict.items():
    filename = name.lower().replace(' ', '_') + '_logistic.pkl'
    filepath = os.path.join(models_path, filename)
    with open(filepath, 'wb') as f:
        pickle.dump(model_pipeline, f)
    print(f"Saved {name} model to {filepath}")

## Key Parameters

| Parameter | Description | Values |
|:----------|:------------|:-------|
| `penalty` | Regularization type | 'l1', 'l2', 'elasticnet' |
| `C` | Inverse regularization strength | float > 0 |
| `solver` | Optimization algorithm | 'saga' for L1/ElasticNet |
| `l1_ratio` | ElasticNet mixing | 0-1 (1=L1, 0=L2) |